In [266]:
import torch

In [267]:
words = open('names.txt').read().splitlines()

In [268]:
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [269]:
new_words = []
for i in range(len(words)):
    new_words.append("." + words[i] + ".")

In [270]:
new_words[:3]

['.emma.', '.olivia.', '.ava.']

In [271]:
for word in new_words[:3]:
    for i in range(len(word)-2):
        print(f"Input -> {word[i], word[i+1]}, Output -> {word[i+2]}")
    print("\n")

Input -> ('.', 'e'), Output -> m
Input -> ('e', 'm'), Output -> m
Input -> ('m', 'm'), Output -> a
Input -> ('m', 'a'), Output -> .


Input -> ('.', 'o'), Output -> l
Input -> ('o', 'l'), Output -> i
Input -> ('l', 'i'), Output -> v
Input -> ('i', 'v'), Output -> i
Input -> ('v', 'i'), Output -> a
Input -> ('i', 'a'), Output -> .


Input -> ('.', 'a'), Output -> v
Input -> ('a', 'v'), Output -> a
Input -> ('v', 'a'), Output -> .




In [272]:
N = torch.zeros((27*27, 27), dtype=torch.int32)

In [273]:
all_letters = sorted(set(''.join(new_words)))
print(all_letters)

['.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [274]:
itos, stoi = {}, {}

for number, letter in enumerate(all_letters):
    stoi[number] = letter
    itos[letter] = number

In [275]:
print(stoi)
print(itos)

{0: '.', 1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z'}
{'.': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26}


In [276]:
for word in new_words:
    for i in range(len(word)-2):
        row = 27*itos[word[i]] + itos[word[i+1]]
        column = itos[word[i+2]]
        N[row, column] = N[row, column] + 1

In [277]:
N

tensor([[  0,   0,   0,  ...,   0,   0,   0],
        [  0, 207, 190,  ...,  27, 173, 152],
        [  0, 169,   0,  ...,   0,   4,   0],
        ...,
        [  1,   0,   0,  ...,   0,   0,   0],
        [ 34,  27,   0,  ...,   0,   0,   1],
        [  4,  13,   0,  ...,   0,   7,   0]], dtype=torch.int32)

In [278]:
N = N + 1

In [279]:
N

tensor([[  1,   1,   1,  ...,   1,   1,   1],
        [  1, 208, 191,  ...,  28, 174, 153],
        [  1, 170,   1,  ...,   1,   5,   1],
        ...,
        [  2,   1,   1,  ...,   1,   1,   1],
        [ 35,  28,   1,  ...,   1,   1,   2],
        [  5,  14,   1,  ...,   1,   8,   1]], dtype=torch.int32)

In [280]:
torch.set_printoptions(sci_mode=False, precision=6)

In [281]:
N[48]

tensor([ 16,   1,  36,   1,  50,   2,   1,  28,   2,   3,   1,   4,  28,   5,
         67,   1,   1,   1, 109,  26,  14,   1,   1,   1,   6,   1,   1],
       dtype=torch.int32)

In [282]:
P = N / N.sum(1, keepdim=True)

In [283]:
P

tensor([[0.037037, 0.037037, 0.037037,  ..., 0.037037, 0.037037, 0.037037],
        [0.000225, 0.046879, 0.043047,  ..., 0.006311, 0.039216, 0.034483],
        [0.000750, 0.127532, 0.000750,  ..., 0.000750, 0.003751, 0.000750],
        ...,
        [0.071429, 0.035714, 0.035714,  ..., 0.035714, 0.035714, 0.035714],
        [0.201149, 0.160920, 0.005747,  ..., 0.005747, 0.005747, 0.011494],
        [0.069444, 0.194444, 0.013889,  ..., 0.013889, 0.111111, 0.013889]])

In [284]:
P.shape

torch.Size([729, 27])

## Counting approach

In [140]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):

    # Generate the first character
    generated_word = []
    ix = 0
    new_ix = torch.multinomial(P[ix], num_samples = 1, replacement = True, generator = g).item()

    # If the first character is '.', then regenerate until we get a alphabet
    while new_ix == 0:
        new_ix = torch.multinomial(P[ix], num_samples = 1, replacement = True, generator = g).item()
        
    generated_word.append(stoi[new_ix])

    # Generate the second character
    ix = 0 * 27 + new_ix    
    new_ix = torch.multinomial(P[ix], num_samples = 1, replacement = True, generator = g).item()

    # If the second character is '.', then print the word generated so far (i.e it will be a single character only)
    if new_ix == 0:
        print(''.join(generated_word))

    # Continue with generation
    else:
        generated_word.append(stoi[new_ix])
        while True:

            # Since we need 2 characters to predict the third, this is the way to calculate ix
            ix = itos[generated_word[-2]]*27 + itos[generated_word[-1]]
            
            new_ix = torch.multinomial(P[ix], num_samples = 1, replacement = True, generator = g).item()
            
            if new_ix == 0:
                break
            generated_word.append(stoi[new_ix]) 
        print(''.join(generated_word))

ce
za
zogh
uriana
kaydnevonimittain
luwak
ka
da
samiyah
javer


In [122]:
loss = []
for word in new_words:
    for i in range(len(word)-2):
        row = itos[word[i]]*27 + itos[word[i+1]]
        col = itos[word[i+2]]
        l = -P[row, col].log()
        loss.append(l)
total_loss = torch.tensor(loss).mean()
print(total_loss)

tensor(2.093080)


## Neural Network approach

In [294]:
import torch.nn.functional as F

In [295]:
x, y = [], []

for word in new_words:
    for i in range(len(word)-2):
        x.append(itos[word[i]]*27 + itos[word[i+1]])
        y.append(itos[word[i+2]])
        
x = torch.tensor(x)
y = torch.tensor(y)

xs = F.one_hot(x, num_classes=729).float()
ys = F.one_hot(y, num_classes=27)

In [296]:
print(xs.shape)
print(ys.shape)

torch.Size([196113, 729])
torch.Size([196113, 27])


In [297]:
W = torch.randn((729, 27), requires_grad=True)


for i in range(200):

    # Forward pass
    z = (xs @ W).exp()
    P = z / z.sum(1, keepdim=True)

    # Loss on forward pass
    loss = -P[torch.arange(196113), y].log().mean()

    # Backpropogation
    W.grad = None
    loss.backward()

    W.data = W.data - 200*W.grad

    if i % 20 == 0:
        print(loss)

tensor(3.738135, grad_fn=<NegBackward0>)
tensor(2.448247, grad_fn=<NegBackward0>)
tensor(2.297299, grad_fn=<NegBackward0>)
tensor(2.233807, grad_fn=<NegBackward0>)
tensor(2.198319, grad_fn=<NegBackward0>)
tensor(2.175482, grad_fn=<NegBackward0>)
tensor(2.159518, grad_fn=<NegBackward0>)
tensor(2.147702, grad_fn=<NegBackward0>)
tensor(2.138581, grad_fn=<NegBackward0>)
tensor(2.131308, grad_fn=<NegBackward0>)


In [298]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):

    # Generate the first character
    generated_word = []
    
    ix = 0
    x = F.one_hot(torch.tensor(ix), num_classes = 729).float().unsqueeze(0)
    z = (x @ W).exp()
    
    P = z / z.sum(1, keepdim=True)
    new_ix = torch.multinomial(P, num_samples = 1, replacement = True, generator = g).item()
    
    while new_ix == 0:
        new_ix = torch.multinomial(P, num_samples = 1, replacement = True, generator = g).item()
        
    generated_word.append(stoi[new_ix])
    
    # (., 'first_char')
            
    # Generate the second character
    ix = 0 * 27 + new_ix
    x = F.one_hot(torch.tensor(ix), num_classes = 729).float().unsqueeze(0)
    z = (x @ W).exp()
    P = z / z.sum(1, keepdim=True)
    
    new_ix = torch.multinomial(P, num_samples = 1, replacement = True, generator = g).item()
    
    # If the second character is '.', then print the word generated so far (i.e it will be a single character only)
    if new_ix == 0:
        print(''.join(generated_word))
    
     # Continue with generation
    else:
        generated_word.append(stoi[new_ix])
        while True:
    
            # Since we need 2 characters to predict the third, this is the way to calculate ix
            ix = itos[generated_word[-2]]*27 + itos[generated_word[-1]]
            x = F.one_hot(torch.tensor(ix), num_classes=729).unsqueeze(0).float()
    
            z = (x @ W).exp()
            P = z / z.sum(1, keepdim=True)
            
            new_ix = torch.multinomial(P, num_samples = 1, replacement = True, generator = g).item()
                
            if new_ix == 0:
                break
            generated_word.append(stoi[new_ix]) 
        print(''.join(generated_word))

dexza
romakuriana
vaydemmilistona
urakayk
ka
ur
romiyah
javer
iotai
is


### Exercise 4 - dropping F.one_hot and indexing into rows of W & Exercise 5 - using F.cross_entropy

In [299]:
import torch.nn.functional as F

In [300]:
x, y = [], []

for word in new_words:
    for i in range(len(word)-2):
        x.append(itos[word[i]]*27 + itos[word[i+1]])
        y.append(itos[word[i+2]])
        
x = torch.tensor(x)
y = torch.tensor(y)

In [301]:
W = torch.randn((729, 27), requires_grad=True)


for i in range(200):

    # Forward pass
    z = W[x]
    
    # Loss on forward pass
    loss = F.cross_entropy(z, y)
    
    # Backpropogation
    W.grad = None
    loss.backward()

    W.data = W.data - 200*W.grad

    if i % 20 == 0:
        print(loss)

tensor(3.835137, grad_fn=<NllLossBackward0>)
tensor(2.454184, grad_fn=<NllLossBackward0>)
tensor(2.297414, grad_fn=<NllLossBackward0>)
tensor(2.233237, grad_fn=<NllLossBackward0>)
tensor(2.197569, grad_fn=<NllLossBackward0>)
tensor(2.174635, grad_fn=<NllLossBackward0>)
tensor(2.158580, grad_fn=<NllLossBackward0>)
tensor(2.146701, grad_fn=<NllLossBackward0>)
tensor(2.137552, grad_fn=<NllLossBackward0>)
tensor(2.130283, grad_fn=<NllLossBackward0>)


In [302]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):

    # Generate the first character
    generated_word = []
    
    ix = 0
    z = W[ix].exp().unsqueeze(0)
    
    P = z / z.sum(1, keepdim=True)
    new_ix = torch.multinomial(P, num_samples = 1, replacement = True, generator = g).item()
    
    while new_ix == 0:
        new_ix = torch.multinomial(P, num_samples = 1, replacement = True, generator = g).item()
        
    generated_word.append(stoi[new_ix])
    
    # (., 'first_char')
            
    # Generate the second character
    ix = 0 * 27 + new_ix
    
    z = W[ix].exp().unsqueeze(0)
    P = z / z.sum(1, keepdim=True)
    
    new_ix = torch.multinomial(P, num_samples = 1, replacement = True, generator = g).item()
    
    # If the second character is '.', then print the word generated so far (i.e it will be a single character only)
    if new_ix == 0:
        print(''.join(generated_word))
    
     # Continue with generation
    else:
        generated_word.append(stoi[new_ix])
        while True:
    
            # Since we need 2 characters to predict the third, this is the way to calculate ix
            ix = itos[generated_word[-2]]*27 + itos[generated_word[-1]]
            
            z = W[ix].exp().unsqueeze(0)
            P = z / z.sum(1, keepdim=True)
            
            new_ix = torch.multinomial(P, num_samples = 1, replacement = True, generator = g).item()
                
            if new_ix == 0:
                break
            generated_word.append(stoi[new_ix])
        print(''.join(generated_word))

cexza
zogpkuriana
vaydemmilistona
noluwan
ka
za
romiyah
javer
vitzi
moriellavojkwgtoda
